In [17]:
from preamble import *
from event_data_conversion import get_time_between_events, events_to_rt_pred
import pandas as pd
import psycopg2
import os
import lightgbm as lgb
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Connecting to the database
con = psycopg2.connect(dbname='mimic', user='postgres', password='Merci', host='localhost', port=5432)  # Database connection credentials
query_schema = 'SET search_path TO mimiciii;'  # Setting the search path schema

# # From the algorithm of Reijnders, K. (2020).
# Prediction of septic shock mortality using deep learning and probabilistic fuzzy systems
# (Master's thesis, Eindhoven University of Technology).
# Retrieved from https://github.com/KevinReijnders/MSc-Thesis-Septic-Shock-Mortality-Prediction.
#######################################################################################################
# Defining paths for HDF files
data_base_path_bm = 'C:\\Users\\bmessaoudi\\PycharmProjects\\SHAP CAUSALITY\\cleaned_BM_events'
data_base_path_lt = 'C:\\Users\\bmessaoudi\\PycharmProjects\\SHAP CAUSALITY\\cleaned_LT_events'

# Loading datasets from the "Event Tables Filtering" notebook
df_bm = pd.read_hdf(os.path.join(data_base_path_bm, "cleaned_BM_events.hdf"))
df_lt = pd.read_hdf(os.path.join(data_base_path_lt, "cleaned_LT_events.hdf"))

# Drop the 'at3' variable because it was only measured 4 times in total
df_lt = df_lt[~(df_lt['variable'] == 'at3')]

# Drop all events with 'valuenum' = NaN as they cannot be used
df_bm = df_bm[~df_bm['valuenum'].isnull()]
df_lt = df_lt[~df_lt['valuenum'].isnull()]

# Concatenate all events into a single dataframe
df = pd.concat([df_bm, df_lt]).reset_index(drop=True)

# Ensure 'charttime' is of type datetime
df['charttime'] = pd.to_datetime(df['charttime'])

# First dataset for modeling
df_mod1 = events_to_rt_pred(df=df[['subject_id', 'charttime', 'variable', 'valuenum']],
                            max_cft=pd.Timedelta(days=200 * 365),
                            verbose=True)
#########################################################################################################
 
def extract_TOD(database_connection):
    """
    Extracts table with the times of death of the septic shock patients.
    Returns the result as a dataframe.

    inputs
    -----
    database_connection: database connection to the MIMIC-III PostgreSQL database.
    """
    
    qry = """ 
    
        WITH ss_patients AS (
        SELECT subject_id, hadm_id, septic_shock
        FROM explicit_sepsis
        WHERE septic_shock = 1
        ),
        prescriptions_with_treatment AS (
        SELECT p.subject_id, COUNT(DISTINCT p.drug) AS num_drugs
        FROM prescriptions p
        WHERE p.drug IN ('Cefazolin', 'Cefepime', 'Ceftazidime', 'Ciprofloxacin', 'Clindamycin', 'Erythromycin',
                     'Gentamicin', 'Levofloxacin', 'Metronidazole', 'Moxifloxacin', 'Piperacillin', 'Rifampin',
                     'Tobramycin', 'Vancomycin', 'Amikacin', 'Ampicillin', 'Azithromycin', 'Aztreonam')
        GROUP BY p.subject_id
        ),
        prescriptions_and_deathtime AS (
        SELECT a.subject_id, MAX(a.deathtime) AS deathtime
        FROM admissions AS a
        INNER JOIN ss_patients AS ssp ON a.subject_id = ssp.subject_id AND a.hadm_id = ssp.hadm_id
        GROUP BY a.subject_id
        )
        SELECT pwt.subject_id, CASE WHEN pwt.num_drugs > 1 THEN '0' ELSE '1' END AS "A", pd.deathtime
        FROM prescriptions_with_treatment AS pwt
        JOIN prescriptions_and_deathtime pd ON pd.subject_id = pwt.subject_id
        GROUP BY pwt.subject_id, "A", pd.deathtime;

        """
    df = pd.read_sql(qry, database_connection, parse_dates='deathtime')

    # Convert boolean values to integers
    df['A'] = df['A'].astype(int)

    return df

df_tod = extract_TOD(con)

df_mod1 = df_mod1.merge(df_tod, on='subject_id', how='left')
df_mod1['time to death'] = df_mod1['deathtime'] - df_mod1['charttime']

# Add the label we are interested in: mortality within 72 hours
df_mod1['y'] = (df_mod1['time to death'] <= pd.Timedelta('72 hours')).astype(int).values

####################################

# Module 1: Calculating the Number of Missing Values in A
missing_A_count = df_mod1['A'].isna().sum()
print(f"Nombre de valeurs manquantes dans 'A' : {missing_A_count}")

# Module 2: Imputing Missing Values in A with LightGBM
if missing_A_count > 0:
    # Data Preparation
    features = df_mod1.drop(columns=['A', 'deathtime', 'time to death', 'y', 'subject_id', 'charttime']).columns
    X = df_mod1[features]
    y = df_mod1['A']
    
    # Separating Known/Missing Data
    known_mask = y.notna()
    X_train = X[known_mask]
    y_train = y[known_mask].astype(int)
    X_missing = X[~known_mask]

    # Model Training with Cross-Validation
    model = lgb.LGBMClassifier()
    
    # Cross-Validation with KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='accuracy')
    
    print(f"Accuracy moyenne après validation croisée : {cv_scores.mean():.4f}")
    
    # Final Training on the Entire Training Dataset
    model.fit(X_train, y_train)
    
    # Prédiction & imputation
    df_mod1.loc[~known_mask, 'A'] = model.predict(X_missing)
    print(f"Valeurs imputées : {X_missing.shape[0]}")
    
    # Validation
    new_missing_count = df_mod1['A'].isna().sum()
    print(f"Valeurs manquantes après imputation : {new_missing_count}")
else:
    print("Aucune valeur manquante à imputer dans 'A'")

# Module 3: Mortality Prédiction with LightGBM and Evaluation
# Préparation des données pour la prédiction de la mortalité
X = df_mod1.drop(columns=['deathtime', 'time to death', 'y', 'subject_id', 'charttime'])
y = df_mod1['y']

# Separating the Data into Training and Test Sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Training the LightGBM Model
model = lgb.LGBMClassifier()
model.fit(X_train, y_train)

# Prediction on the Test Set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Model Evaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

#####################################

# Randomly split train-val-test 70%-15%-15% based on patients
from sklearn.model_selection import train_test_split

random_state = 10  # Trial & error with random state to achieve approximately 70%-15%-15%

# Select all subjects and whether they have y=1 eventually
all_subjects = df_mod1['subject_id'].unique()
all_subj_labels = df_mod1.groupby('subject_id')['y'].max().sort_index().values

# Make a split among subjects, stratified on the target label
# First set aside 30%, then split that 30% 50-50 to get the desired 70%-15%-15%
train_subj, test_subj, train_subj_labels, test_subj_labels = train_test_split(
    all_subjects,
    all_subj_labels,
    test_size=0.3,
    random_state=random_state,
    stratify=all_subj_labels)

val_subj, test_subj = train_test_split(
    test_subj,
    test_size=0.5,
    random_state=random_state,
    stratify=test_subj_labels)

# Split the instances for modeling based on the subjects
df_train, df_val, df_test = [df_mod1[df_mod1['subject_id'].isin(train_subj)],
                             df_mod1[df_mod1['subject_id'].isin(val_subj)],
                             df_mod1[df_mod1['subject_id'].isin(test_subj)]]

# Check if split conforms to requirements
print("Patients in train, val, test sets:", len(train_subj), len(val_subj), len(test_subj))
print("Observations in train, val, test sets:", len(df_train), len(df_val), len(df_test))
print("% of total observations in train, val, test sets",
      len(df_train) / len(df_mod1) * 100,
      len(df_val) / len(df_mod1) * 100,
      len(df_test) / len(df_mod1) * 100, "\n")
print('Fraction y=1 in train, val, test sets:', df_train['y'].mean(), df_val['y'].mean(), df_test['y'].mean())

#################

# Re-sort all datasets based on instances & time
df_train = df_train.sort_values(['subject_id', 'charttime']).reset_index(drop=True)
df_val = df_val.sort_values(['subject_id', 'charttime']).reset_index(drop=True)
df_test = df_test.sort_values(['subject_id', 'charttime']).reset_index(drop=True)

# Storing train-val-test features & labels in numpy format
feat_colnames = df_train.columns[2:-3]
X_train, X_val, X_test = df_train[feat_colnames].values, \
    df_val[feat_colnames].values, \
    df_test[feat_colnames].values
y_train, y_val, y_test = df_train['y'].values, df_val['y'].values, df_test['y'].values

# Checking shapes
print(X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape) 

Filling variable 0: arterial base excess
Filling variable 1: arterial pco2
Filling variable 2: arterial ph
Filling variable 3: arterial po2
Filling variable 4: bicarbonate
Filling variable 5: bilirubin
Filling variable 6: calcium
Filling variable 7: creatinine
Filling variable 8: crp
Filling variable 9: cvp
Filling variable 10: diastolic blood pressure
Filling variable 11: fio2
Filling variable 12: glucose
Filling variable 13: got(asat)
Filling variable 14: gpt(alat)
Filling variable 15: heart rate
Filling variable 16: hematocrit
Filling variable 17: platelets
Filling variable 18: potassium
Filling variable 19: ptt
Filling variable 20: sodium
Filling variable 21: spo2
Filling variable 22: systolic blood pressure
Filling variable 23: temperature
Filling variable 24: urea
Filling variable 25: white blood cells


C:\Users\bmessaoudi\AppData\Local\Temp\2\ipykernel_13228\832521837.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(qry, database_connection, parse_dates='deathtime')


Nombre de valeurs manquantes dans 'A' : 19872
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 204450, number of negative: 519526
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009598 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5889
[LightGBM] [Info] Number of data points in the train set: 723976, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.282399 -> initscore=-0.932593
[LightGBM] [Info] Start training from score -0.932593
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 204819, number of negative: 519158
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009979 seconds.
You can set `force_row_wise=true` to remove the overhead

In [46]:
# ------------------------- Definition of columns -------------------------
feat_colnames = df_train.columns[2:-3]

# ------------------------- Extraction of A -------------------------
A_train = df_train['A'].copy()
A_val = df_val['A'].copy()
A_test = df_test['A'].copy()

# ------------------------- Data préparation -------------------------
X_train = df_train[feat_colnames]
X_val = df_val[feat_colnames]
X_test = df_test[feat_colnames]

# ------------------------- Transformation -------------------------
# Step 1: Imputation of missing values
si = SimpleImputer(strategy='mean')
X_train = si.fit_transform(X_train)
X_val = si.transform(X_val)
X_test = si.transform(X_test)

# Step 2: Data Normalization
ss = StandardScaler()
X_train = ss.fit_transform(X_train)
X_val = ss.transform(X_val)
X_test = ss.transform(X_test)

# ------------------------- Reconstruction -------------------------
def rebuild_df(base_df, transformed, A_col):
    df = pd.DataFrame(transformed, columns=feat_colnames, index=base_df.index)
    df['A'] = A_col
    return df

df_mod1_train = rebuild_df(df_train, X_train, A_train)
df_mod1_val = rebuild_df(df_val, X_val, A_val)
df_mod1_test = rebuild_df(df_test, X_test, A_test)

# ------------------------- Vérification -------------------------
def check_binarite(*dfs):
    """Vérifie si A est binaire dans tous les datasets"""
    checks = [df['A'].isin([0, 1]).all() for df in dfs]
    return all(checks)

print("\nColonne A binaire partout :", 
      check_binarite(df_mod1_train, df_mod1_val, df_mod1_test))

# ------------------------- Calculation of occurrences of A -------------------------
def count_A_values(df):
    """Compte le nombre d'occurrences de A=0, A=1 et calcule le total"""
    count_0 = (df['A'] == 0).sum()
    count_1 = (df['A'] == 1).sum()
    total = df['A'].count()  
    return count_0, count_1, total

for nom, dataframe in zip(["Train"], [df_mod1_train, df_mod1_val, df_mod1_test]):
    cnt0, cnt1, total = count_A_values(dataframe)
    print(f"Dataset {nom} : A=0: {cnt0}, A=1: {cnt1}, Total: {total}")
    
# ------------------------- Descriptive statistics -------------------------
print("\nStatistiques des caractéristiques pour chaque dataset :")
for nom, dataframe in zip(["Train"], [df_mod1_train, df_mod1_val, df_mod1_test]):
    print(f"\nDataset {nom}:")
    print(dataframe[feat_colnames].describe().transpose()[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])    

# Renaming datasets with new variables
X_train, X_val, X_test = df_mod1_train, df_mod1_val, df_mod1_test


Colonne A binaire partout : True
Dataset Train : A=0: 482269, A=1: 189739, Total: 672008

Statistiques des caractéristiques pour chaque dataset :

Dataset Train:
                             count          mean       std        min  \
arterial base excess      672008.0  3.165684e-17  1.000001  -5.475721   
arterial pco2             672008.0 -6.228806e-16  1.000001  -2.548670   
arterial ph               672008.0  3.662741e-14  1.000001 -12.945243   
arterial po2              672008.0  3.268881e-16  1.000001  -1.999821   
bicarbonate               672008.0 -1.217720e-15  1.000001  -2.963134   
bilirubin                 672008.0 -1.024354e-16  1.000001  -0.479009   
calcium                   672008.0 -3.161159e-15  1.000001  -7.687818   
creatinine                672008.0 -1.650300e-16  1.000001  -1.221527   
crp                       672008.0 -7.222691e-16  1.000001  -5.219437   
cvp                       672008.0 -7.332965e-16  1.000001  -3.495059   
diastolic blood pressure  672008.0

In [47]:
# Store means & stds used for imputation to recover original scale later
means = ss.mean_
stds = np.sqrt(ss.var_)

# Brief check : is the mean now 0 and std now 1
pd.DataFrame(X_train).describe()

,arterial base excess,arterial pco2,arterial ph,arterial po2,bicarbonate,bilirubin,calcium,creatinine,crp,cvp,diastolic blood pressure,fio2,glucose,got(asat),gpt(alat),heart rate,hematocrit,platelets,potassium,ptt,sodium,spo2,systolic blood pressure,temperature,urea,white blood cells,A
count,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,6.720080e+05,672008.000000
mean,3.165684e-17,-6.228806e-16,3.662741e-14,3.268881e-16,-1.217720e-15,-1.024354e-16,-3.161159e-15,-1.650300e-16,-7.222691e-16,-7.332965e-16,1.150632e-15,-3.349344e-16,5.967854e-16,-2.325097e-17,-1.209283e-16,8.578538e-16,2.994098e-15,-3.738764e-17,-2.285150e-15,-4.700523e-16,-9.446722e-15,3.313881e-15,1.269551e-15,-5.794534e-15,-5.285022e-16,1.026891e-16,0.282346
std,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,1.000001e+00,0.450141
min,-5.475721e+00,-2.548670e+00,-1.294524e+01,-1.999821e+00,-2.963134e+00,-4.790092e-01,-7.687818e+00,-1.221527e+00,-5.219437e+00,-3.495059e+00,-2.978372e+00,-3.099957e+00,-2.253128e+00,-1.986744e-01,-2.769366e-01,-4.855621e+00,-6.713742e+00,-1.380333e+00,-4.516831e+00,-1.239212e+00,-7.067114e+00,-1.550399e+01,-4.587828e+00,-7.937110e+00,-1.267623e+00,-1.355570e+00,0.000000
25%,-6.475374e-01,-6.386427e-01,-5.835133e-01,-5.371692e-01,-6.372355e-01,-4.098961e-01,-6.593320e-01,-6.757436e-01,-9.041286e-16,-5.416601e-16,-6.668267e-01,-6.426294e-01,-5.826598e-01,-1.679936e-01,-2.270948e-01,-7.120794e-01,-6.787901e-01,-7.698864e-01,-6.836541e-01,-5.706151e-01,-6.540223e-01,-3.490707e-01,-6.878859e-01,-5.415384e-01,-6.831796e-01,-5.743863e-01,0.000000
50%,1.841894e-02,-1.080795e-01,6.032682e-02,-1.081246e-01,-1.277530e-01,-3.580613e-01,-6.369762e-02,-3.725306e-01,-9.041286e-16,-5.416601e-16,-8.894052e-02,-2.829743e-02,-1.739281e-01,-1.437720e-01,-1.872214e-01,-3.023073e-02,-7.296479e-02,-1.975923e-01,-1.836745e-01,-3.111596e-01,-7.101394e-02,1.755225e-01,-1.022056e-01,-3.678669e-03,-2.935509e-01,-1.998461e-01,0.000000
75%,5.178862e-01,3.163710e-01,7.041669e-01,3.404220e-01,5.045692e-01,-2.977420e-02,5.319368e-01,4.158231e-01,-9.041286e-16,-5.416601e-16,5.659972e-01,-8.730182e-16,3.414292e-01,-6.949233e-02,-6.427827e-02,7.040678e-01,5.328605e-01,5.400313e-01,6.496248e-01,1.428876e-01,7.063305e-01,6.126836e-01,6.052044e-01,5.341811e-01,3.465535e-01,3.780159e-01,1.000000
max,5.179581e+00,1.058277e+01,4.824744e+00,1.028646e+01,4.014562e+00,1.372373e+01,2.280866e+01,8.105304e+00,4.288930e+00,6.872478e+00,7.038323e+00,3.043362e+00,2.343921e+01,3.664565e+01,2.764776e+01,6.473556e+00,7.499851e+00,7.935343e+00,1.364909e+01,5.421809e+00,8.285439e+00,8.166921e-01,4.094022e+00,6.854033e+00,5.439558e+00,4.814795e+01,1.000000


In [48]:
# Use of the mean of the training set for the test as well : to avoid leakage
pd.DataFrame(X_train).mode()

,arterial base excess,arterial pco2,arterial ph,arterial po2,bicarbonate,bilirubin,calcium,creatinine,crp,cvp,diastolic blood pressure,fio2,glucose,got(asat),gpt(alat),heart rate,hematocrit,platelets,potassium,ptt,sodium,spo2,systolic blood pressure,temperature,urea,white blood cells,A
0,0.018419,0.0,3.431069e-14,2.771405e-16,-0.172056,-0.444453,0.055429,-0.675744,-9.041286e-16,-5.416601e-16,-0.127466,-0.028297,-0.138386,-4.589468e-17,-9.443928e-17,-0.659629,-0.422479,1.716414,-0.350334,-1.063582e-15,0.317658,0.816692,2.741700e-15,-0.205376,-0.488365,-0.60649,0.0


In [49]:
# Compute standalone sample weights (without class weights)
instance_weights = []

for dfx in [df_train, df_val, df_test]:
    # Dataframe with sample weights, initially contains counts (number of instances) per patient
    dfx_weights = dfx\
        .groupby('subject_id')['charttime']\
        .count()\
        .to_frame()\
        .reset_index()\
        .rename(columns={'charttime':'instance_weight'}) # To be sure to pick column without missing values

    # Compute weights from the counts based on the number of instances per patient
    dfx_weights['instance_weight'] = dfx_weights['instance_weight'].apply(lambda x: 1/x)

    # Converting sample weights to the shape that fits X, i.e. row-per-instance
    instance_weight = dfx.merge(dfx_weights, on='subject_id', how='left')['instance_weight'].values
    
    # Store result for use outside this loop
    instance_weights.append(instance_weight)
    
instance_weights_train, instance_weights_val, instance_weights_test = instance_weights

In [51]:
import os
import pandas as pd
import numpy as np

# ------------------------- Save Directory Configuration -------------------------
# Define the base path for saving files
data_base_path = 'C:\\Users\\bmessaoudi\\PycharmProjects\\SHAP CAUSALITY\\'
os.makedirs(data_base_path, exist_ok=True)  # Create the directory if it doesn't exist

# Dataframes
for dfx, name in zip([df_train, df_val, df_test], ['df_train', 'df_val', 'df_test']):
    dfx.to_hdf(path_or_buf=os.path.join(data_base_path, "modeling_data2/" + name + ".hdf"),
               key='Processed_data',
               mode='w',
               complevel=9
            )

# Processed features & target labels
for X_, y_, X_name, y_name in zip([X_train, X_val, X_test], [y_train, y_val, y_test], ['X_train', 'X_val', 'X_test'], ['y_train', 'y_val', 'y_test']):
    pd.DataFrame(X_).to_hdf(
        path_or_buf=os.path.join(data_base_path, "modeling_data2/" + X_name + ".hdf"),
        key='Processed_data',
        mode='w',
        complevel=9
    )
    
    pd.DataFrame(y_).to_hdf(
        path_or_buf=os.path.join(data_base_path, "modeling_data2/" + y_name + ".hdf"),
        key='Processed_data',
        mode='w',
        complevel=9
    )

# Means & stds for recovering original scale
pd.DataFrame(means).to_hdf(
    path_or_buf=os.path.join(data_base_path, "modeling_data2/" + 'means' + ".hdf"),
    key='Processed_data',
    mode='w',
    complevel=9
)

pd.DataFrame(stds).to_hdf(
    path_or_buf=os.path.join(data_base_path, "modeling_data2/" + 'stds' + ".hdf"),
    key='Processed_data',
    mode='w',
    complevel=9
)

# Instance weights
for instance_weights, name in zip([instance_weights_train, instance_weights_val, instance_weights_test], ['instance_weights_train', 'instance_weights_val', 'instance_weights_test']):
    pd.DataFrame(instance_weights).to_hdf(
        path_or_buf=os.path.join(data_base_path, "modeling_data2/" + name + ".hdf"),
        key='Processed_data',
        mode='w',
        complevel=9
    )